# Algoritmo Genético — exemplo binário do começo ao fim

Este notebook mostra o programa funcionando com o dataset binário (`dataset_example_1.xlsx`, classes 0 e 1).

1. **Passo a passo:** roda uma geração na mão, chamando cada função na ordem do algoritmo
2. **Programa completo:** usa `rodar()` para treinar e `prever()` para classificar dados novos

## 0. Imports

In [1]:
import sys
sys.path.insert(0, "../../source")  # pasta onde fica o pacote algoritmo_genetico

import numpy as np

from algoritmo_genetico.io import ler_excel
from algoritmo_genetico.criar_crom import criar_cromossomos
from algoritmo_genetico.fitness import fitness_binario, fitness_percentual
from algoritmo_genetico.selecao import selecionar_pais_roleta
from algoritmo_genetico.cruzamento import cruzar_um_ponto
from algoritmo_genetico.mutacao import mutar
from algoritmo_genetico.atualizar_crom import atualizar_populacao
from algoritmo_genetico import rodar, prever

np.random.seed(0)  # semente: rodando de novo, os sorteios saem iguais

## 1. Lendo os dados

`ler_excel` ignora a primeira coluna (nome das lições), usa a última como classe e o resto como features.

In [2]:
X, y = ler_excel("dataset_example_1.xlsx")

print("X (amostras, features):", X.shape)
print("y (amostras,):", y.shape)
print("classe 0:", sum(y == 0), "| classe 1:", sum(y == 1))
print("\nprimeira amostra:", X[0], "-> classe", y[0])

X (amostras, features): (539, 18)
y (amostras,): (539,)
classe 0: 261 | classe 1: 278

primeira amostra: [1 0 0 1 1 0 0 1 0 1 1 0 0 1 1 0 1 0] -> classe 1


## 2. Separando treino e teste

80% dos dados para treinar e 20% para testar. O teste mostra se o modelo acerta dados que **nunca viu**.

`np.random.permutation(n)` devolve os números de 0 a n-1 embaralhados, e usamos isso como ordem das amostras.

In [3]:
ordem = np.random.permutation(len(y))
corte = int(0.8 * len(y))

X_treino, y_treino = X[ordem[:corte]], y[ordem[:corte]]
X_teste, y_teste = X[ordem[corte:]], y[ordem[corte:]]

print("treino:", X_treino.shape, "| teste:", X_teste.shape)

treino: (431, 18) | teste: (108, 18)


---
# Parte 1 — Uma geração passo a passo

### Etapa 1: `criar_cromossomos`
Cada cromossomo tem 1 gene de bias e 1 peso para cada feature.

In [4]:
qtd_genes = X_treino.shape[1] + 1  # features + bias
populacao = criar_cromossomos(qtd_cromossomos=6, qtd_genes=qtd_genes)

print("populacao:", populacao.shape)
print("primeiro cromossomo:", populacao[0].round(2))

populacao: (6, 19)
primeiro cromossomo: [ 0.24  0.73 -0.68  0.98 -0.83 -0.81  0.27  0.9   0.87 -0.11  0.46  0.48
 -0.39 -0.76 -0.11 -0.22  0.06  0.69  0.07]


### Etapa 2: `fitness_binario` e `fitness_percentual`
Fitness = recall da classe 1 × recall da classe 0, então vai de 0 a 1.
Depois os fitnesses viram as chances de cada cromossomo na roleta.

In [5]:
fitnesses = fitness_binario(populacao, X_treino, y_treino)
percentuais = fitness_percentual(fitnesses)

for i in range(len(populacao)):
    print(f"cromossomo {i}: fitness = {fitnesses[i]:.4f} | chance na roleta = {percentuais[i]:.2%}")

cromossomo 0: fitness = 0.0827 | chance na roleta = 10.69%
cromossomo 1: fitness = 0.2062 | chance na roleta = 26.67%
cromossomo 2: fitness = 0.2204 | chance na roleta = 28.50%
cromossomo 3: fitness = 0.0876 | chance na roleta = 11.33%
cromossomo 4: fitness = 0.0266 | chance na roleta = 3.44%
cromossomo 5: fitness = 0.1498 | chance na roleta = 19.37%


### Etapa 3: `selecionar_pais_roleta`
Quem tem fitness maior tem mais chance de ser escolhido como pai ou mãe.

In [6]:
pai, mae = selecionar_pais_roleta(populacao, percentuais)

print("pai:", pai.round(2))
print("mae:", mae.round(2))

pai: [-0.74  0.42  0.42  0.22  0.34 -0.1   0.39 -0.24 -0.8   0.92 -0.86 -0.96
  0.96 -0.7  -0.48 -0.57  0.16  0.95 -0.45]
mae: [ 0.24  0.73 -0.68  0.98 -0.83 -0.81  0.27  0.9   0.87 -0.11  0.46  0.48
 -0.39 -0.76 -0.11 -0.22  0.06  0.69  0.07]


### Etapa 4: `cruzar_um_ponto`
O filho pega o começo do pai e, a partir de um ponto de corte aleatório, o final da mãe.

In [7]:
filhos = cruzar_um_ponto(pai, mae, qtd_filhos=3)

for i, filho in enumerate(filhos):
    print(f"filho {i}:", filho.round(2))

filho 0: [-0.74  0.42  0.42  0.22  0.34 -0.1   0.39 -0.24  0.87 -0.11  0.46  0.48
 -0.39 -0.76 -0.11 -0.22  0.06  0.69  0.07]
filho 1: [-0.74  0.42  0.42  0.22  0.34 -0.1   0.39 -0.24 -0.8   0.92 -0.86  0.48
 -0.39 -0.76 -0.11 -0.22  0.06  0.69  0.07]
filho 2: [-0.74  0.42  0.42  0.22 -0.83 -0.81  0.27  0.9   0.87 -0.11  0.46  0.48
 -0.39 -0.76 -0.11 -0.22  0.06  0.69  0.07]


### Etapa 5: `mutar`
Alguns genes de cada filho são trocados por um valor aleatório entre -1 e 1.

In [8]:
filhos_mutados = mutar(filhos, qtd_genes_mutados=1)

for i in range(len(filhos)):
    mudou = [g for g in range(qtd_genes) if filhos[i][g] != filhos_mutados[i][g]]
    print(f"filho {i}: gene(s) mutado(s) = {mudou}")

filho 0: gene(s) mutado(s) = [1]
filho 1: gene(s) mutado(s) = [15]
filho 2: gene(s) mutado(s) = [15]


### Etapa 6: `atualizar_populacao`
Os piores cromossomos da população saem e os melhores filhos entram no lugar.

In [9]:
fitnesses_filhos = fitness_binario(filhos_mutados, X_treino, y_treino)
print("fitness dos filhos:", fitnesses_filhos.round(4))

nova_populacao = atualizar_populacao(populacao, fitnesses, filhos_mutados, fitnesses_filhos, qtd_substituicoes=2)
novos_fitnesses = fitness_binario(nova_populacao, X_treino, y_treino)

print("\nfitness antes :", fitnesses.round(4), "| melhor:", round(fitnesses.max(), 4))
print("fitness depois:", novos_fitnesses.round(4), "| melhor:", round(novos_fitnesses.max(), 4))

fitness dos filhos: [0.4515 0.1933 0.2531]

fitness antes : [0.0827 0.2062 0.2204 0.0876 0.0266 0.1498] | melhor: 0.2204
fitness depois: [0.4515 0.2062 0.2204 0.0876 0.2531 0.1498] | melhor: 0.4515


Isso foi **uma geração**. O algoritmo repete as etapas 2 a 6 até atingir o fitness alvo ou acabar as gerações.

---
# Parte 2 — Programa completo

### Treinando com `rodar`
Use `verbose=True` para ver o melhor fitness de cada geração.

In [10]:
np.random.seed(0)

modelo = rodar(
    X_treino, y_treino,
    qtd_cromossomos=6,
    qtd_genes_mutados=1,
    qtd_filhos=3,
    geracoes=500,
    fitness_alvo=0.90,
    verbose=False,
)

print("melhor fitness:", round(modelo["fitness"], 4))
print("parou na geracao:", modelo["geracao"])
print("melhor cromossomo:", modelo["cromossomo"].round(2))

melhor fitness: 0.8216
parou na geracao: 500
melhor cromossomo: [ 0.32 -0.92  0.84 -0.63  0.39  0.63 -0.06  0.47 -0.35 -0.89 -0.46  0.44
  0.01 -0.55  0.75  0.22  0.69 -0.26 -0.41]


### Testando com `prever`
Classifica as amostras de teste e compara com a classe certa.

In [11]:
previsto = prever(modelo, X_teste)

acertos = sum(previsto == y_teste)
print(f"acuracia no teste: {acertos}/{len(y_teste)} = {acertos / len(y_teste):.2%}")

# matriz de confusao: quantas de cada classe real foram previstas como 0 ou 1
print("\n               previu 0  previu 1")
for real in [0, 1]:
    p0 = sum((y_teste == real) & (previsto == 0))
    p1 = sum((y_teste == real) & (previsto == 1))
    print(f"real {real}       {p0:>8}  {p1:>8}")

acuracia no teste: 94/108 = 87.04%

               previu 0  previu 1
real 0             48         3
real 1             11        46


### Classificando uma amostra só

In [12]:
amostra = X_teste[0]
print("amostra:", amostra)
print("classe prevista:", prever(modelo, amostra)[0], "| classe real:", y_teste[0])

amostra: [1 0 0 1 1 0 0 1 0 1 1 0 0 1 1 0 1 0]
classe prevista: 1 | classe real: 1
